In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import pandas as pd

df = pd.read_csv('/content/drive/MyDrive/chi_tnp_sql_output/bigqoutput.csv')

In [ ]:
!ls /content/drive/MyDrive/chi_tnp_sql_output/

In [ ]:
df.head()

In [ ]:
df.shape

In [ ]:
def label_signature(late_night_dev, weekend_dev, short_dev):
  if late_night_dev > 20 and weekend_dev > 15 and short_dev > 0:
    return 'Nightlife Pocket'
  elif late_night_dev > 10 and weekend_dev > 8 and short_dev > -5:
    return 'Leans Nightlife Pocket'
  elif late_night_dev < 0 and weekend_dev < 0 and short_dev < 0:
    return 'Commuter Zone'
  elif late_night_dev <= 10 and weekend_dev <= 8 and short_dev < 5:
    return 'Leans Commuter Zone'
  else:
    return 'Mixed/Other'

In [ ]:
label_signature(25,20,5)

In [ ]:
df['signature'] = df.apply(lambda row: label_signature(row['late_night_dev'], row['weekend_dev'], row['short_dev']), axis=1)

In [ ]:
df['signature'].value_counts()

In [ ]:
# grouping up coords

In [ ]:
df['cluster_lat'] = df['pickup_lat_grid'].round(2)
df['cluster_lon'] = df['pickup_lon_grid'].round(2)
df.groupby(['cluster_lat', 'cluster_lon']).size().sort_values(ascending=False)

In [ ]:
pd.set_option('display.max_rows', None)
pd.set_option('display.max_colwidth', None)

In [ ]:
def has_conflict(df, lat_column, lon_column):
  clustered = df.groupby([lat_column, lon_column])['signature'].unique().reset_index()
  return clustered[clustered['signature'].apply(lambda x: len(x) > 1)]

In [ ]:
has_conflict(df, 'cluster_lat', 'cluster_lon')

In [ ]:
s= 0.005
df['cluster_lat_500m'] = round(df['pickup_lat_grid'] / s ) * s
df['cluster_lon_500m'] = round(df['pickup_lon_grid'] / s ) * s

In [ ]:
df[['cluster_lat_500m', 'cluster_lon_500m']].head()

In [ ]:
has_conflict(df, 'cluster_lat_500m', 'cluster_lon_500m')

In [ ]:
s= 0.004
df['cluster_lat_400m'] = round(df['pickup_lat_grid'] / s ) * s
df['cluster_lon_400m'] = round(df['pickup_lon_grid'] / s ) * s

In [ ]:
df[['cluster_lat_400m', 'cluster_lon_400m']].head()

In [ ]:
has_conflict(df, 'cluster_lat_400m', 'cluster_lon_400m')

In [28]:
df.to_csv('/content/drive/MyDrive/chi_tnp_sql_output/final_clustered_output.csv', index=False)